In [ ]:
"""
Just a function to search for keywords in the repo to help navigate module functions
"""
import os
import json

# File types to search
VALID_EXTENSIONS = {".py", ".ipynb", ".txt", ".yaml", ".yml", ".json"}


ROOT_DIR = os.getcwd()       # 
SEARCH_WORD = "windows_per_utterance"    # Word to search for
CASE_SENSITIVE = False  

def line_contains(line, word):
    if CASE_SENSITIVE:
        return word in line
    return word.lower() in line.lower()


def search_in_file(filepath, search_word):
    matches = []

    try:
        if filepath.endswith(".ipynb"):
            with open(filepath, "r", encoding="utf-8") as f:
                notebook = json.load(f)
                for cell in notebook.get("cells", []):
                    if cell.get("cell_type") in ["code", "markdown"]:
                        for i, line in enumerate(cell.get("source", []), 1):
                            if line_contains(line, search_word):
                                matches.append((i, line.strip()))
        else:
            with open(filepath, "r", encoding="utf-8") as f:
                for i, line in enumerate(f, 1):
                    if line_contains(line, search_word):
                        matches.append((i, line.strip()))

    except Exception:
        pass  # Skip unreadable files

    return matches


def search_repo(root_dir, search_word):
    results = {}

    for root, _, files in os.walk(root_dir):
        for file in files:
            ext = os.path.splitext(file)[1]
            if ext in VALID_EXTENSIONS:
                filepath = os.path.join(root, file)
                matches = search_in_file(filepath, search_word)
                if matches:
                    results[filepath] = matches

    return results


In [2]:
results = search_repo(ROOT_DIR, SEARCH_WORD)

if results:
    print(f"\nFound '{SEARCH_WORD}' in {len(results)} file(s):\n")
    for filepath, matches in results.items():
        print(filepath)
        for line_no, line in matches:
            print(f"  Line {line_no}: {line}")
        print()
else:
    print(f"No matches found for '{SEARCH_WORD}'.")


Found 'windows_per_utterance' in 2 file(s):

/work/DISCOURSE/contextual-redundancy/Search_contextual-redundancy.ipynb
  Line 10: SEARCH_WORD = "windows_per_utterance"    # Word to search for

/work/DISCOURSE/contextual-redundancy/src/data/custom_pickle_datamodule.py
  Line 29: windows_per_utterance: int = 4,
  Line 38: self.windows_per_utterance = int(windows_per_utterance)
  Line 152: return len(self.processed_data) * self.windows_per_utterance
  Line 155: base_idx = idx // self.windows_per_utterance
  Line 257: windows_per_utterance: int = 4,
  Line 271: self.windows_per_utterance = windows_per_utterance
  Line 304: windows_per_utterance=self.windows_per_utterance,
  Line 313: windows_per_utterance=self.windows_per_utterance,
  Line 323: windows_per_utterance=self.windows_per_utterance,



In [ ]:
"""
Quick function for checking for duplicate windows in the data
"""

import pickle
import hashlib
import numpy as np
from collections import Counter
from pathlib import Path

PKL_PATH = Path("./Data/prosodic_features_splits_CLEANED/relative_prominence/train.pkl")

MAX_CONTEXT_WORDS = 10
WINDOWS_PER_UTTERANCE = 4
SEED = 12345


def hash_window(tokens, labels):
    labels_arr = np.asarray(labels, dtype=float)
    labels_str = ",".join([f"{x:.5f}" for x in labels_arr.flatten()])
    s = " ".join(tokens) + "|" + labels_str
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def build_random_windows(words, labels, max_context_words, windows_per_utt, rng):
    n = len(words)
    windows = []

    if n == 0:
        return windows

    for _ in range(windows_per_utt):
        context_length = int(rng.integers(1, min(max_context_words, n) + 1))
        start = int(rng.integers(0, n - context_length + 1))
        end = start + context_length

        windows.append((words[start:end], labels[start:end]))

    return windows


def check_duplicates(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    rng = np.random.default_rng(SEED)

    all_hashes = []
    total_windows = 0

    for text, labels in zip(data["texts"], data["labels"]):
        words = str(text).split()

        if len(words) != len(labels):
            continue

        windows = build_random_windows(
            words,
            labels,
            MAX_CONTEXT_WORDS,
            WINDOWS_PER_UTTERANCE,
            rng,
        )

        for w_tokens, w_labels in windows:
            all_hashes.append(hash_window(w_tokens, w_labels))

        total_windows += len(windows)

    counter = Counter(all_hashes)
    duplicates = sum(v - 1 for v in counter.values() if v > 1)

    print(f"Total windows:     {total_windows}")
    print(f"Unique windows:    {len(counter)}")
    print(f"Duplicates:        {duplicates}")
    print(f"Duplicate ratio:   {duplicates / total_windows:.4f}")

    print("\nTop duplicate counts:")
    for _, count in counter.most_common(10):
        if count <= 1:
            break
        print(count)

    return counter


counter = check_duplicates(PKL_PATH)

Total windows:     63608
Unique windows:    62674
Duplicates:        934
Duplicate ratio:   0.0147

Top duplicate counts:
4
4
3
3
3
3
3
3
3
3
